In [2]:
from dotenv import load_dotenv
import os
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import precision_recall_curve, average_precision_score, confusion_matrix, ConfusionMatrixDisplay
from models.TabTransformer import run_experiment as run_experiment_c, infer_tabtransformer as infer_tabtransformer_c
from models.qCLSTabTransformer import run_experiment as run_experiment_q, infer_tabtransformer as infer_tabtransformer_q
from utils.Preprocessor import preprocess_amex
from utils.CheckpointHandler import checkpoint_from_model, model_from_checkpoint

%load_ext autoreload
%autoreload 2

In [3]:
#Use .env file with Kaggle API Token to download AMEX dataset (Don't run this if you want to use the UGB dataset)
load_dotenv()
os.environ["KAGGLE_API_TOKEN"] = os.getenv("KAGGLE_API_TOKEN")
path = kagglehub.competition_download("amex-default-prediction")
file = os.path.join(path, "train_data.csv")
data_all = pd.read_csv(file, nrows=1e6)
file = os.path.join(path, "train_labels.csv")
labels_all = pd.read_csv(file,nrows=1e6)

#Scrape all data associated with the a random n customers
customers = data_all["customer_ID"].drop_duplicates().sample(n=12000, random_state=42)
data_without_labels = data_all[data_all["customer_ID"].isin(customers)]
labels = labels_all[labels_all["customer_ID"].isin(customers)]
data = preprocess_amex(data_without_labels, labels)
frac_default = np.sum(data['Class'])/len(data['Class'])

print(f"Size of dataset: {len(data)}")
print(f"Number of features: {data.shape[1]}")
print(f"# of defaults = {int(np.sum(data['Class']))}")
print(frac_default*100, '% default')

Size of dataset: 12000
Number of features: 226
# of defaults = 3124
26.033333 % default


In [4]:
#Split data into default and non-default
default_data = data[data['Class'] == 1].sample(frac=1, random_state=42)
non_default_data = data[data['Class'] == 0].sample(frac=1, random_state=42)

train_size = 2000

training_default = int(frac_default*train_size)
data_subset = pd.concat([default_data[:training_default], non_default_data[:train_size - training_default]]).sample(frac=1, random_state=42)

In [ ]:
#Run classical experiment on small dataset
model_c, test_results_c, test_df = run_experiment_c(data_subset, lr=3e-3)
torch.save(checkpoint_from_model(model_c), "../checkpoints/classical_model_amex.pt")
print(f'Classical model test Amex metric: {test_results_c["amex_metric"]}')

#Run quantum experiment on small dataset
model_q, test_results_q, test_df = run_experiment_q(data_subset, lr=3e-3, qufex_params=(4,2,2))
torch.save(checkpoint_from_model(model_q), "../checkpoints/quantum_model_amex(4,2,2).pt")
print(f'Quantum CLS model test Amex metric: {test_results_q["amex_metric"]}')

In [ ]:
#Run this for pretrained models
model_c = model_from_checkpoint(torch.load("../checkpoints/classical_model_amex.pt", map_location="cpu", weights_only=True))
model_q = model_from_checkpoint(torch.load("../checkpoints/quantum_model_amex(4,2,2).pt", map_location="cpu", weights_only=True))

In [7]:
#Tabulate total parameters and training parameters to see model size
total_params_c = sum(p.numel() for p in model_c.parameters())
trainable_params_c = sum(p.numel() for p in model_c.parameters() if p.requires_grad)

print("Classical: ")
print(f"Total params: {total_params_c:,}")
print(f"Trainable params: {trainable_params_c:,}")

total_params_q = sum(p.numel() for p in model_q.parameters())
trainable_params_q = sum(p.numel() for p in model_q.parameters() if p.requires_grad)

print("\nQuantum: ")
print(f"Total params: {total_params_q:,}")
print(f"Trainable params: {trainable_params_q:,}")

Classical: 
Total params: 32,673
Trainable params: 32,673

Quantum: 
Total params: 32,969
Trainable params: 32,969


In [8]:
#Gathers metrics for different inference sizes to see how well the models generalize
inference_sizes = [2000, 4000, 6000, 8000, 10000]
amex_metrics_c = []
amex_metrics_q = []
default_recalls_c = []
default_recalls_q = []
non_default_specificities_c = []
non_default_specificities_q = []


inference_default = int(frac_default * np.max(inference_sizes))
inference_df = pd.concat([default_data[training_default:training_default + inference_default],
                            non_default_data[train_size - training_default:train_size - training_default + np.max(inference_sizes) - inference_default]]
                            ).sample(frac=1, random_state=42)

#Classical model inference
results = infer_tabtransformer_c(model_c, inference_df)
preds_c_all = results['preds']
probs_c_all = results['probs']
targets_c_all = results['targets']

#Quantum model inference
results = infer_tabtransformer_q(model_q, inference_df)
preds_q_all = results['preds']
probs_q_all = results['probs']
targets_q_all = results['targets']

#Calculates inference metrics by size grouping to visualize scaling
for size in inference_sizes:
    preds_c = preds_c_all[:size]
    probs_c = probs_c_all[:size]
    targets_c = targets_c_all[:size]
    default_preds_c = preds_c[targets_c == 1]       #Collects predictions for true default data
    non_default_preds_c = preds_c[targets_c == 0]   #Collects predictions for true non-default data
    default_scores_c = probs_c[targets_c == 1]      #Collects scores for true default data
    non_default_scores_c = probs_c[targets_c == 0]  #Collects scores for true non-default data

    amex_metrics_c.append(average_precision_score(targets_c, probs_c))
    default_recalls_c.append(np.sum(default_preds_c) / len(default_preds_c))                           #TP / (TP + FN)
    non_default_specificities_c.append(1 - np.sum(non_default_preds_c) / len(non_default_preds_c))     #TN / (TN + FP)

    preds_q = preds_q_all[:size]
    probs_q = probs_q_all[:size]
    targets_q = targets_q_all[:size]
    default_preds_q = preds_q[targets_q == 1]       #Collects predictions for true default data
    non_default_preds_q = preds_q[targets_q == 0]   #Collects predictions for true non-default data
    default_scores_q = probs_q[targets_q == 1]      #Collects scores for true default data
    non_default_scores_q = probs_q[targets_q == 0]  #Collects scores for true non-default data

    amex_metrics_q.append(average_precision_score(targets_q, probs_q))
    default_recalls_q.append(np.sum(default_preds_q) / len(default_preds_q))                           #TP / (TP + FN)
    non_default_specificities_q.append(1 - np.sum(non_default_preds_q) / len(non_default_preds_q))     #TN / (TN + FP)

In [ ]:
plt.hist(default_scores_c, bins=30, alpha=0.5, density=True, label="default")
plt.hist(non_default_scores_c, bins=30, alpha=0.5, density=True, label="Non-default")
plt.axvline(model_c.threshold, linestyle="--", label=f"Threshold = {model_c.threshold:.3f}")
plt.title("Score Distribution (Classical)")
plt.xlabel("Predicted Default Probability")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
plt.hist(default_scores_q, bins=30, alpha=0.5, density=True, label="default")
plt.hist(non_default_scores_q, bins=30, alpha=0.5, density=True, label="Non-default")
plt.axvline(model_q.threshold, linestyle="--", label=f"Threshold = {model_q.threshold:.3f}")
plt.title("Score Distribution (Quantum)")
plt.xlabel("Predicted Defualt Probability")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
plt.plot(inference_sizes, amex_metrics_c, label="Classical", color='b')
plt.plot(inference_sizes, amex_metrics_q, label="Quantum", color='r')
plt.xlabel("Inference Dataset Size")
plt.ylabel("Amex metric")
plt.legend()
plt.show()

In [ ]:
plt.plot(inference_sizes, default_recalls_c, label="Classical", color='b')
plt.plot(inference_sizes, default_recalls_q, label="Quantum", color='r')
plt.xlabel("Inference Dataset Size")
plt.ylabel("default Recall")
plt.legend()
plt.show()

In [ ]:
plt.plot(inference_sizes, non_default_specificities_c, label="Classical", color='b')
plt.plot(inference_sizes, non_default_specificities_q, label="Quantum", color='r')
plt.xlabel("Inference Dataset Size")
plt.ylabel("Non-default Specificities")
plt.legend()
plt.show()

In [ ]:
#Two level grid search for optimal learning rate (using same dataset size as before)
lrs = [1e-5, 1e-4, 1e-3, 1e-2]
best_amex_metric = 0
best_lr = 0
for lr in lrs:
    test_amex_metrics = []
    for seed in range(5):
        data_subset = pd.concat([default_data.sample(n=training_default, random_state=seed),
                                  non_default_data.sample(n=train_size-training_default, random_state=seed)]
                                  ).sample(frac=1, random_state=seed)
        
        model, test_results, test_df = run_experiment_q(data_subset, lr=lr, qufex_params=(4,2,2))
        test_amex_metric = test_results['amex_metric']
        test_amex_metrics.append(test_amex_metric)

    mean_amex_metric = np.mean(test_amex_metrics)
    if mean_amex_metric > best_amex_metric:
        best_amex_metric = mean_amex_metric
        best_lr = lr
    print(mean_amex_metric, lr)

print()

lrs = [best_lr*.3, best_lr*0.7, best_lr*3, best_lr*7]
for lr in lrs:
    test_amex_metrics = []
    for seed in range(5):
        data_subset = pd.concat([default_data.sample(n=training_default, random_state=seed),
                                    non_default_data.sample(n=train_size-training_default, random_state=seed)]
                                    ).sample(frac=1, random_state=seed)
        
        model, test_results, test_df = run_experiment_q(data_subset, lr=lr, qufex_params=(4,2,2))
        test_amex_metric = test_results['amex_metric']
        test_amex_metrics.append(test_amex_metric)

    mean_amex_metric = np.mean(test_amex_metrics)
    if mean_amex_metric > best_amex_metric:
        best_amex_metric = mean_amex_metric
        best_lr = lr
    print(mean_amex_metric, lr)

print(f'\nBest LR: {best_lr}')